# ATP Tennis Match Analysis — Exploratory Data Analysis

**CIP Team 107 | School Project**

This notebook explores a dataset of ATP professional tennis matches.  
We examine:
- Surface and tournament distributions
- Player rankings and win rates
- Age profiles of winners vs losers
- Rolling win-rate trends

If you have real ATP match data (e.g. from [Jeff Sackmann's tennis_atp repo](https://github.com/JeffSackmann/tennis_atp)), place the CSV files in `../data/` and run `load_multiple()`. Otherwise the demo uses synthetic data.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import generate_sample_data
from src.features import compute_player_stats
from src.visualizations import (
    plot_surface_distribution,
    plot_rank_vs_win_rate,
    plot_age_distribution,
    plot_win_rate_by_surface,
    plot_top_players,
)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
print('Libraries loaded.')

## 1. Load Data

In [ ]:
# --- Option A: use synthetic demo data (no file needed) ---
df = generate_sample_data(n_matches=3000, seed=42)

# --- Option B: load real ATP data ---
# from src.data_loader import load_multiple
# import glob
# files = sorted(glob.glob('../data/atp_matches_*.csv'))
# df = load_multiple(files)

print(f'Loaded {len(df):,} matches')
df.head()

## 2. Dataset Overview

In [ ]:
print('Shape:', df.shape)
print('\nData types:\n', df.dtypes)
print('\nMissing values:\n', df.isnull().sum()[df.isnull().sum() > 0])

In [ ]:
df[['winner_rank', 'loser_rank', 'winner_age', 'loser_age', 'minutes']].describe().round(2)

## 3. Surface & Tournament Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

plot_surface_distribution(df, ax=axes[0])

# Tournament distribution
top_tourneys = df['tourney_name'].value_counts().head(10)
top_tourneys[::-1].plot(kind='barh', ax=axes[1], color='steelblue', edgecolor='white')
axes[1].set_title('Top 10 Tournaments by Match Count', fontsize=13)
axes[1].set_xlabel('Match Count')

plt.tight_layout()
plt.show()

## 4. Player Rankings

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(df['winner_rank'].dropna(), bins=40, alpha=0.7, color='steelblue', label='Winner')
axes[0].hist(df['loser_rank'].dropna(), bins=40, alpha=0.7, color='tomato', label='Loser')
axes[0].set_title('Rank Distribution: Winners vs Losers', fontsize=13)
axes[0].set_xlabel('ATP Rank')
axes[0].set_ylabel('Frequency')
axes[0].legend()

rank_diff = df['loser_rank'] - df['winner_rank']
axes[1].hist(rank_diff.dropna(), bins=40, color='mediumpurple', edgecolor='white')
axes[1].axvline(0, color='black', linestyle='--', linewidth=1)
axes[1].set_title('Rank Difference (Loser − Winner)', fontsize=13)
axes[1].set_xlabel('Rank Difference')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

upset_pct = (rank_diff < 0).mean() * 100
print(f'Upset rate (lower-ranked player wins): {upset_pct:.1f}%')

## 5. Rolling Win Rates

In [ ]:
df_stats = compute_player_stats(df, window=20)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
plot_rank_vs_win_rate(df_stats, ax=axes[0])
plot_win_rate_by_surface(df_stats, ax=axes[1])
plt.tight_layout()
plt.show()

## 6. Age Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

plot_age_distribution(df, ax=axes[0])

# Average winner age by surface
avg_age = df.groupby('surface')['winner_age'].mean().sort_values()
avg_age.plot(kind='barh', ax=axes[1], color='steelblue', edgecolor='white')
axes[1].set_title('Average Winner Age by Surface', fontsize=13)
axes[1].set_xlabel('Age')

plt.tight_layout()
plt.show()

## 7. Top Players

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
plot_top_players(df, top_n=15, ax=ax)
plt.tight_layout()
plt.show()

## 8. Correlation Heatmap

In [ ]:
numeric_cols = ['winner_rank', 'loser_rank', 'winner_age', 'loser_age', 'minutes']
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', ax=ax, linewidths=0.5)
ax.set_title('Correlation Matrix of Numeric Features', fontsize=13)
plt.tight_layout()
plt.show()

## Key Findings

- **Surface distribution**: Hard courts dominate ATP scheduling (~55% of matches).
- **Rank advantage**: Higher-ranked (lower rank number) players win ~65% of matches — strong baseline signal.
- **Age**: Winners are, on average, slightly younger than losers; both distributions peak around 25–28.
- **Win-rate vs rank**: A positive correlation exists — better-ranked players also maintain higher rolling win rates.

These findings motivate the features used in the prediction model (Notebook 02).